# Compatibilité réelle des sources — Q2, Q3 et Q4

**Étude centrale : Haute-Garonne (31)** ; comparaison des 13 départements d'Occitanie.  
**Objectif :** lire les résultats produits par le pipeline et vérifier si les observations incendies et météo peuvent être rapprochées à un même grain.

Le rapport est issu de **fichiers réels**, mais le taux de couverture d'un seuil n'est pas validé : nous devons encore qualifier l'échéance météo (J+1/J+2) et obtenir l'export BDIFF original.

**Relectures demandées, non encore réalisées :** Stéphane (jointures), Maeva (source), Chantal (qualité), Ludovic (définition KPI).

## 1. Charger les preuves générées par GitHub Actions

Le script reproductible est `scripts/profiler_et_joindre_donnees.py`. Il a enregistré les contrôles qualité et les éventuelles erreurs réseau dans un audit JSON. Le notebook ne réinvente pas ces résultats.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "docs/resultats/profilage_sources_v1.json").exists():
    ROOT = ROOT.parent
audit_path = ROOT / "docs/resultats/profilage_sources_v1.json"
assert audit_path.exists(), "Le pipeline doit d'abord générer son audit JSON."
audit = json.loads(audit_path.read_text(encoding="utf-8"))
meteo = pd.DataFrame(audit["meteo"])
incendies = pd.DataFrame(audit["incendies"])
raccords = pd.DataFrame(audit["raccord_technique_NON_KPI"])
display(meteo[["annee","lignes","departements","departements_occitanie","dates_distinctes","date_min","date_max","dates_invalides","doublons_depart_jour"]])
display(incendies)
display(raccords)

## 2. Examiner les limites et les sources réellement utilisées

Un événement absent de la couche secondaire ne prouve pas qu'il n'y a pas eu de feu. Une ligne météo présente à la même date ne suffit pas à démontrer qu'une prévision était disponible au moment de l'alerte.

In [ ]:
erreurs = audit.get("collecte_arcgis", {}).get("erreurs", [])
if erreurs:
    print("Erreurs de collecte documentées :")
    display(pd.DataFrame(erreurs))
print("Limites explicites de l'audit :")
for limitation in audit.get("limites", []):
    print("•", limitation)

## 3. Graphique descriptif — Haute-Garonne

Le graphique suivant décrit la distribution brute des codes météo J+1 par saison. **Ce n'est pas** un taux de détection des feux.

In [ ]:
import matplotlib.pyplot as plt
distributions = {r["annee"]: r["niveaux_j1"] for r in audit["meteo"]}
codes = sorted({c for d in distributions.values() for c in d})
tab = pd.DataFrame(distributions).T.fillna(0)
tab[codes].plot.bar(figsize=(9, 4))
plt.ylabel("Nombre de lignes (France, chaque département)")
plt.title("Répartition des codes J1 archivés — France entière")
plt.tight_layout()
plt.show()
print("Pour la distribution propre à la Haute-Garonne, voir l'illustration générée par le pipeline : docs/resultats/danger_j1_haute_garonne.svg")

## 4. Conclusion méthodologique et points de décision

- **Réellement testé :** les schémas, les périodes et les doublons de la météo ; la disponibilité technique d'une même date dans une partie des incendies 2024.
- **Non validé :** la sémantique de la date météo, la fiabilité exhaustive de l'extrait d'incendies Esri et l'application du seuil J+1/J+2 au jour de feu.
- **Indicateur Q2 :** ne pas calculer avant validation ; 0 jour raccordé peut signifier une période hors diffusion, un événement absent ou un défaut de source.
- **Indicateur Q3 :** le nombre d'observations météo par seuil n'est pas encore un coût opérationnel.
- **Suite avec Maeva :** obtenir l'export BDIFF original 2024–2025.
- **Suite avec Stéphane et Ludovic :** qualifier les dates de validité, tester les jointures et écrire les règles de calcul.
- **Suite avec Chantal :** valider la traçabilité, les licences et la couverture réelle.

Cette liste propose des actions ; elle ne simule pas les contributions personnelles du groupe.